In [1]:
print('hello world')

hello world


In [2]:
import os
from datetime import datetime, timedelta

base_dir = "/Users/galaxy/codebase"
result_dir = "/Users/galaxy/codebase/results/data_extract"
os.makedirs(result_dir, exist_ok=True)

# 1. 추출 기간 설정 (2026-01-01 ~ 2026-02-28)
start_date = datetime(2026, 1, 1)
end_limit = datetime(2026, 2, 28) + timedelta(days=1)

# 2. 파일 이름 동적 생성 (예: 0101_0228_gbike_revenue_extract.xlsx)
prefix = f"{start_date.strftime('%m%d')}_{end_limit.strftime('%m%d')}"
result_file = os.path.join(result_dir, f"{prefix}_gbike_revenue_extract.xlsx")

# 4일 단위로 기간 생성
date_ranges = []
current_start = start_date
while current_start < end_limit:
    current_end = min(current_start + timedelta(days=4), end_limit)
    date_ranges.append((current_start.strftime('%Y-%m-%d %H:%M:%S'), current_end.strftime('%Y-%m-%d %H:%M:%S')))
    current_start = current_end

In [3]:
date_ranges

[('2026-01-01 00:00:00', '2026-01-05 00:00:00'),
 ('2026-01-05 00:00:00', '2026-01-09 00:00:00'),
 ('2026-01-09 00:00:00', '2026-01-13 00:00:00'),
 ('2026-01-13 00:00:00', '2026-01-17 00:00:00'),
 ('2026-01-17 00:00:00', '2026-01-21 00:00:00'),
 ('2026-01-21 00:00:00', '2026-01-25 00:00:00'),
 ('2026-01-25 00:00:00', '2026-01-29 00:00:00'),
 ('2026-01-29 00:00:00', '2026-02-02 00:00:00'),
 ('2026-02-02 00:00:00', '2026-02-06 00:00:00'),
 ('2026-02-06 00:00:00', '2026-02-10 00:00:00'),
 ('2026-02-10 00:00:00', '2026-02-14 00:00:00'),
 ('2026-02-14 00:00:00', '2026-02-18 00:00:00'),
 ('2026-02-18 00:00:00', '2026-02-22 00:00:00'),
 ('2026-02-22 00:00:00', '2026-02-26 00:00:00'),
 ('2026-02-26 00:00:00', '2026-03-01 00:00:00')]

In [ ]:
from tqdm.auto import tqdm

bar = tqdm(range(1000))
for i in bar:
    bar.set_description(f'현재 숫자: {i}')

In [9]:
import pandas as pd
import sqlalchemy
from sqlalchemy import create_engine
import urllib.parse
import os
import sys
from datetime import datetime, timedelta

file_name = 'data_extract.csv'
# 1. 추출 기간 설정 (2026-01-01 ~ 2026-02-28)
start_date = datetime(2025, 1, 1)
end_limit = datetime(2025, 12, 31) + timedelta(days=1)

def get_engine():
    host = 'live.st.rds.gbility.io'
    user = 'gbikemarketing'
    password = urllib.parse.quote_plus('gbikemkt0514$@#!')
    port = 3306
    database = 'gbike'
    try:
        engine = create_engine(f"mysql+pymysql://{user}:{password}@{host}:{port}/{database}")
        return engine
    except Exception as e:
        print(f"Engine creation failed: {e}")
        sys.exit(1)

def main(file_name,start_date,end_limit):
    base_dir = "/Users/galaxy/codebase"
    result_dir = "/Users/galaxy/codebase/results/data_extract"
    os.makedirs(result_dir, exist_ok=True)


    # 2. 파일 이름 동적 생성 (예: 0101_0228_gbike_revenue_extract.xlsx)
    prefix = f"{start_date.strftime('%m%d')}_{end_limit.strftime('%m%d')}"
    result_file = os.path.join(result_dir, f"{prefix}_{file_name}.csv")

    engine = get_engine()


    query = f"""
    select o.user_id,o.from_api,count(o.order_id) as ord_cnt
    from gbike.rich_orders o
    where 1=1
    AND o.add_time >= UNIX_TIMESTAMP('{start_date}') - 32400
    AND o.add_time < UNIX_TIMESTAMP('{end_limit}') - 32400
    and o.order_state = 2
    group by o.user_id, o.from_api
    """
    with engine.connect() as connection:
        df = pd.read_sql(query, connection)
    df.to_excel(result_file, index=False, engine='openpyxl')
    print(f"Final file saved: {result_file} (Total: {len(df)} rows)")

if __name__ == "__main__":
    main(file_name,start_date,end_limit)

KeyboardInterrupt: 

In [2]:
%%sql
    WITH region AS (
        SELECT
            r.region_id,
            r.region_name   AS small_area,
            rr.region_name  AS mid_area,
            rrr.region_name AS large_area
        FROM gbike.rich_region r
        JOIN gbike.rich_region rr  ON r.parent_id  = rr.region_id
        JOIN gbike.rich_region rrr ON rr.parent_id = rrr.region_id
        WHERE rr.region_name = '강릉캠프'
    ),
    user_base AS (
        SELECT
            ru.user_id,
            ru.region_id,
            CASE
                WHEN MOD(ru.gender, 2) = 1 THEN '남자'
                WHEN ru.gender IS NOT NULL THEN '여자'
                ELSE 'unknown'
            END AS gender,
            CASE
                WHEN ru.birthday IS NULL THEN NULL
                ELSE FLOOR(
                    (EXTRACT(YEAR FROM CURRENT_DATE) * 10000
                     + EXTRACT(MONTH FROM CURRENT_DATE) * 100
                     + EXTRACT(DAY FROM CURRENT_DATE)
                     - ru.birthday) / 10000
                )
            END AS age
        FROM gbike.rich_user ru
    ),
    user_info AS (
        SELECT
            ub.*,
            CASE
                WHEN ub.age IS NULL       THEN 'unknown'
                WHEN ub.age < 16          THEN '16세미만'
                WHEN ub.age < 20          THEN '16세이상20세미만'
                WHEN ub.age < 30          THEN '20대'
                WHEN ub.age < 40          THEN '30대'
                WHEN ub.age < 50          THEN '40대'
                WHEN ub.age < 60          THEN '50대'
                ELSE '기타'
            END AS age_group
        FROM user_base ub
    )
    SELECT
        DATE_FORMAT(
            CONVERT_TZ(FROM_UNIXTIME(O.add_time), 'UTC', 'Asia/Seoul'),
            '%%Y-W%%v'
        )                                                                       AS week_label,
        DATE_FORMAT(
            CONVERT_TZ(FROM_UNIXTIME(O.add_time), 'UTC', 'Asia/Seoul'),
            '%%Y-%%m'
        )                                                                       AS month_label,
        R.large_area,
        R.mid_area,
        R.small_area,
        CASE
            WHEN O.bicycle_type IN (15, 17, 18, 22, 24) THEN '자전거'
            WHEN O.bicycle_type IN (9, 10, 11, 12, 13, 16, 19, 23) THEN '킥보드'
            ELSE '기타기종'
        END                                                                     AS category,
        UI.gender,
        UI.age_group,
        COUNT(O.order_id)                                                       AS trip_count,
        SUM(
            (CASE WHEN O.from_api = 'gcooter' THEN O.pay_amount
                  ELSE O.order_amount END
             + O.out_of_area_charge) / 1.1
        )                                                                       AS net_revenue
    FROM gbike.rich_orders O
    JOIN user_info UI        ON O.user_id   = UI.user_id
    JOIN region R            ON O.region_id = R.region_id
    WHERE O.order_state = 2
      AND O.add_time BETWEEN
            UNIX_TIMESTAMP(CONVERT_TZ('2025-01-01 00:00:00', 'Asia/Seoul', 'UTC'))
          AND
            UNIX_TIMESTAMP(CONVERT_TZ('2025-01-01 23:59:59',   'Asia/Seoul', 'UTC'))
    GROUP BY
        week_label,
        month_label,
        R.large_area,
        R.mid_area,
        R.small_area,
        category,
        UI.gender,
        UI.age_group

,week_label,month_label,large_area,mid_area,small_area,category,gender,age_group,trip_count,net_revenue
0,%Y-W%v,%Y-%m,중앙RS팀,강릉캠프,(본사직영)_강원_속초,킥보드,남자,16세미만,7,10424.5455
1,%Y-W%v,%Y-%m,중앙RS팀,강릉캠프,(본사직영)_강원_속초,킥보드,남자,16세이상20세미만,68,96011.8186
2,%Y-W%v,%Y-%m,중앙RS팀,강릉캠프,(본사직영)_강원_속초,자전거,남자,16세이상20세미만,34,51838.1819
3,%Y-W%v,%Y-%m,중앙RS팀,강릉캠프,(본사직영)_강원_속초,킥보드,unknown,unknown,2,5418.1818
4,%Y-W%v,%Y-%m,중앙RS팀,강릉캠프,(본사직영)_강원_속초,킥보드,남자,20대,61,84895.4543
...,...,...,...,...,...,...,...,...,...,...
112,%Y-W%v,%Y-%m,중앙RS팀,강릉캠프,(본사직영)_강원_삼척,자전거,남자,기타,1,378.1818
113,%Y-W%v,%Y-%m,중앙RS팀,강릉캠프,(본사직영)_강원_삼척,자전거,여자,16세미만,11,13100.9092
114,%Y-W%v,%Y-%m,중앙RS팀,강릉캠프,(본사직영)_강원_삼척,킥보드,남자,16세미만,8,18312.7273
115,%Y-W%v,%Y-%m,중앙RS팀,강릉캠프,(본사직영)_강원_삼척,자전거,여자,50대,4,7774.5454
